In [1]:
import pandas as pd
import re
from datetime import datetime

In [2]:
customers_df = pd.read_csv("../data/raw/customers.csv")

products_df = pd.read_csv("../data/raw/products.csv")

orders_df = pd.read_csv("../data/raw/orders.csv")

order_items_df = pd.read_csv("../data/raw/order_items.csv")

In [3]:
print(customers_df.shape)
print(products_df.shape)
print(orders_df.shape)
print(order_items_df.shape)

(600, 5)
(500, 5)
(700, 5)
(2500, 6)


In [4]:
def clean_orders(df):
    """
    Cleans the orders dataset by:
    - Standardizing date formats.
    - Filling missing customer IDs.
    """

    cleaned = df.copy()

    # Standardize date formats
    cleaned["order_date"] = pd.to_datetime(
        cleaned["order_date"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

    # Replace missing customer IDs
    cleaned["customer_id"] = cleaned["customer_id"].fillna("UNKNOWN")

    return cleaned

In [5]:
orders_clean = clean_orders(orders_df)

orders_clean.head()

,order_id,customer_id,region,status,order_date
0,ORD00001,CUST0434,South,CANCELLED,2026-03-31 18:36:11
1,ORD00002,CUST0551,South,RETURNED,2026-02-01 12:00:23
2,ORD00003,CUST0171,West,DELIVERED,2025-12-26 06:56:07
3,ORD00004,CUST0144,South,DELIVERED,2025-08-02 16:14:54
4,ORD00005,CUST0462,North,DELIVERED,2025-02-18 15:04:35


In [6]:
orders_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   order_id     700 non-null    str           
 1   customer_id  700 non-null    str           
 2   region       700 non-null    str           
 3   status       700 non-null    str           
 4   order_date   700 non-null    datetime64[us]
dtypes: datetime64[us](1), str(4)
memory usage: 27.5 KB


In [7]:
def clean_products(df):
    """
    Cleans product names by removing extra spaces
    and converting them to Title Case.
    """

    cleaned = df.copy()

    cleaned["product_name"] = (
        cleaned["product_name"]
        .str.strip()
        .str.title()
    )

    return cleaned

In [8]:
products_clean = clean_products(products_df)

products_clean.head()

,product_id,product_name,category,subcategory,cost_price
0,PROD0001,Dining Table,Home,Furniture,691.21
1,PROD0002,Kurti,Clothing,Women,3740.12
2,PROD0003,Jeans,Clothing,Men,3719.16
3,PROD0004,Lamp,Home,Decor,4904.50
4,PROD0005,Dress,Clothing,Women,1868.32


In [9]:
products_df.sample(10)[["product_name"]]

,product_name
498,Kids Shoes
281,Sweater
119,Pressure Cooker
468,Shirt
418,Mystery Novel
487,Batman Comic
216,MacBook Air
296,Dell XPS
319,Office Chair
122,Kids T-Shirt


In [10]:
products_clean.loc[
    products_df.sample(10).index,
    ["product_name"]
]

,product_name
194,Top
260,Kids Jacket
466,Keyboard
495,Kurti
22,Mirror
38,Marvel Comic
324,Hoodie
128,Bookshelf
72,Jeans
392,Galaxy S24


In [11]:
import re

def validate_emails(df):
    """
    Returns customer IDs with invalid email addresses.
    """

    pattern = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

    invalid = df[
        ~df["email"].str.match(pattern, na=False)
    ]

    return invalid["customer_id"].tolist()

In [12]:
invalid_customers = validate_emails(customers_df)

print(invalid_customers)

['CUST0018', 'CUST0088', 'CUST0096', 'CUST0247', 'CUST0248', 'CUST0302', 'CUST0417', 'CUST0459', 'CUST0488', 'CUST0501', 'CUST0503', 'CUST0569']


In [13]:
len(invalid_customers)

12

In [14]:
def check_referential_integrity(orders_df, order_items_df):
    """
    Returns order_items whose order_id
    does not exist in orders.
    """

    valid_orders = set(orders_df["order_id"])

    invalid_rows = order_items_df[
        ~order_items_df["order_id"].isin(valid_orders)
    ]

    return invalid_rows

In [15]:
invalid_order_items = check_referential_integrity(
    orders_df,
    order_items_df
)

invalid_order_items

,order_item_id,order_id,product_id,quantity,unit_price,discount_percent


In [16]:
orders_clean = clean_orders(orders_df)

products_clean = clean_products(products_df)

customers_clean = customers_df.copy()

order_items_clean = order_items_df.copy()

In [18]:
customers_clean.to_csv(
    "../data/cleaned/customers_clean.csv",
    index=False
)

products_clean.to_csv(
    "../data/cleaned/products_clean.csv",
    index=False
)

orders_clean.to_csv(
    "../data/cleaned/orders_clean.csv",
    index=False
)

order_items_clean.to_csv(
    "../data/cleaned/order_items_clean.csv",
    index=False
)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


In [19]:
null_customer_ids = orders_df["customer_id"].isna().sum()

invalid_emails = len(validate_emails(customers_df))

negative_quantities = (order_items_df["quantity"] < 0).sum()

wrong_dates = (
    orders_df["order_date"]
    .str.match(r"\d{2}-\d{2}-\d{4}")
    .sum()
)

invalid_order_refs = len(
    check_referential_integrity(
        orders_df,
        order_items_df
    )
)

report = f"""
DATA QUALITY REPORT
===================

Null Customer IDs      : {null_customer_ids}
Invalid Emails         : {invalid_emails}
Negative Quantities    : {negative_quantities}
Wrong Date Formats     : {wrong_dates}
Broken Order References: {invalid_order_refs}
"""

In [20]:
with open("../output/issues_report.txt", "w") as file:
    file.write(report)

print(report)


DATA QUALITY REPORT

Null Customer IDs      : 35
Invalid Emails         : 12
Negative Quantities    : 75
Wrong Date Formats     : 35
Broken Order References: 0

